In [ ]:
!pip install datasets
!pip install torch
!pip install transformers
!pip install pillow

In [ ]:
from datasets import load_dataset

# Load the dataset
dataset = load_dataset("ashraq/fashion-product-images-small", split="train")

# Display the first record
print(dataset[0])

In [ ]:
import re

def extract_brand(record):
    brand = None
    if record.get('productDisplayName'):
        brand = record['productDisplayName'].split(" ")[0]  # Assuming the first word is the brand

    return brand

# Apply the extraction function to the dataset
dataset = dataset.map(lambda x: {'brand': extract_brand(x)})

# Display the updated dataset
print(dataset[0])

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import base64
from PIL import Image

# Helper function to encode image to base64
def encode_image(image):
    import io
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    return base64.b64encode(buffered.getvalue()).decode("utf-8")

# Filter and transform the dataset
def transform(record):

    if record['brand'] and record['image']:
        return {
            "name": record['productDisplayName'],
            "brand": record['brand'],
            "image": encode_image(record['image'])
        }
    return None

# Apply the transformation
filtered_data = [
    transform(record) for record in dataset
    if transform(record) is not None
]

# Save the filtered dataset as a JSON file
output_path = "/content/drive/My Drive/filtered_dataset-fpis.json"
with open(output_path, "w") as json_file:
    json.dump(filtered_data, json_file, indent=4)

print(f"Filtered dataset has been saved to {output_path}")

In [ ]:
print(filtered_data[0])

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
import pickle
import base64
from PIL import Image
import os
import json
from io import BytesIO

# Path to the filtered dataset JSON file
filtered_dataset_path = "/content/drive/My Drive/filtered_dataset-fpis.json"

# Load the filtered dataset
with open(filtered_dataset_path, "r") as f:
    filtered_data = json.load(f)

# Load the CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

# Helper function to decode base64 image
def decode_base64_image(base64_string):
    try:
        image_data = base64.b64decode(base64_string)
        image = Image.open(BytesIO(image_data)).convert("RGB")
        return image
    except Exception as e:
        print(f"Error decoding base64 image: {e}")
        return None

# Compute embeddings and store metadata
embeddings = []
metadata = []

for i in range(12000, 36000):
    record = filtered_data[i]
    image_base64 = record["image"]
    if image_base64:
        try:
            # Decode the base64 image
            image = decode_base64_image(image_base64)
            if image is None:
                continue

            # Prepare the image for the model
            inputs = processor(images=image, return_tensors="pt")
            with torch.no_grad():
                embedding = model.get_image_features(**inputs).squeeze().numpy()

            # Store the embedding and metadata
            embeddings.append(embedding)
            metadata.append({
                "name": record["name"],
                "brand": record["brand"],
                "image": image_base64
            })
        except Exception as e:
            print(f"Error processing image {image}: {e}")

# Save embeddings and metadata as a pickle file
output_pickle_path = "/content/drive/My Drive/image_embeddings_fpis.pkl"

# Check if the pickle file exists
if os.path.exists(output_pickle_path):
    # Load existing data
    with open(output_pickle_path, "rb") as f:
        existing_data = pickle.load(f)
    existing_embeddings = existing_data.get("embeddings", [])
    existing_metadata = existing_data.get("metadata", [])
else:
    # Initialize empty data if the file doesn't exist
    existing_embeddings = []
    existing_metadata = []

# Append new embeddings and metadata
combined_embeddings = existing_embeddings + embeddings
combined_metadata = existing_metadata + metadata

# Save the updated data back to the pickle file
with open(output_pickle_path, "wb") as f:
    pickle.dump({"embeddings": combined_embeddings, "metadata": combined_metadata}, f)

print(f"Embeddings and metadata have been saved to {output_pickle_path}")


In [ ]:
output_pickle_path = "/content/drive/My Drive/image_embeddings_fpis.pkl"

with open(output_pickle_path, "rb") as f:
    data = pickle.load(f)

print(f"Number of embeddings: {len(data['embeddings'])}")
print(f"Embedding{data['embeddings'][0]}")
print(f"Metadata: {data['metadata'][0]}")


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
from google.colab import files
import base64
from io import BytesIO
import pickle
from IPython.display import display

# Load the CLIP model and processor
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model.eval()

# Predefined brand names for brand prediction
brands = ["Nike", "Adidas", "Puma", "Fastrack", "Levis", "Jockey"]

# Load precomputed embeddings and metadata
pickle_file_path = "/content/drive/My Drive/image_embeddings_fpis.pkl"
with open(pickle_file_path, "rb") as f:
    data = pickle.load(f)

stored_embeddings = data["embeddings"]
stored_metadata = data["metadata"]

# Helper function to decode base64-encoded images
def decode_base64_image(base64_string):
    image_data = base64.b64decode(base64_string)
    return Image.open(BytesIO(image_data)).convert("RGB")

# Helper function to display images with a custom width
def display_image_with_width(image, width):
    aspect_ratio = image.height / image.width
    new_height = int(width * aspect_ratio)
    resized_image = image.resize((width, new_height))
    display(resized_image)

# Step 1: Upload the query image
print("Please upload an image file:")
uploaded = files.upload()
query_image_path = list(uploaded.keys())[0]
query_image = Image.open(query_image_path).convert("RGB")

# Step 2: Predict the brand of the query image
# Process the query image with the predefined brand names
inputs = processor(text=brands, images=query_image, return_tensors="pt", padding=True)

# Extract image and text inputs
image_inputs = inputs["pixel_values"]  # For images
text_inputs = inputs["input_ids"]      # For text

# Compute embeddings for the query image and brand names
with torch.no_grad():
    image_embeds = model.get_image_features(pixel_values=image_inputs)
    text_embeds = model.get_text_features(input_ids=text_inputs)

# Convert embeddings to numpy arrays
image_embeds = image_embeds.cpu().numpy()
text_embeds = text_embeds.cpu().numpy()

# Compute similarity to predict the brand
similarities = cosine_similarity(image_embeds, text_embeds)
predicted_brand = brands[similarities.argmax()]

# Display the query image and the predicted brand
print("\nQuery Image:")
display_image_with_width(query_image, width=350)
print(f"\nPredicted Brand: {predicted_brand}")

# Step 3: Find similar images with the same brand
# Compute embedding for the query image
def compute_query_embedding(image):
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        embedding = model.get_image_features(**inputs).squeeze().numpy()
    return embedding

query_embedding = compute_query_embedding(query_image)

# Compute cosine similarity with stored embeddings
similarities = cosine_similarity([query_embedding], stored_embeddings)[0]

# Sort indices by similarity
sorted_indices = similarities.argsort()[::-1]  # Descending order

# Filter images with the predicted brand
filtered_indices = [index for index in sorted_indices if stored_metadata[index]['brand'] == predicted_brand]

# Get the top 5 similar images with the predicted brand
top_indices = filtered_indices[:5]

# Step 4: Display the top 5 similar images
print("\nTop 5 Similar Images with the Predicted Brand:")
for i, index in enumerate(top_indices):
    meta = stored_metadata[index]
    print(f"\nMatch {i+1}:")
    print(f"  Name: {meta['name']}")
    decoded_image = decode_base64_image(meta['image'])
    display_image_with_width(decoded_image, width=300)